<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  LoRA Ranks Experiment



In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.13/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


In [7]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

ranks = [4, 8, 16, 32]
rank_results = []

for rank in ranks:

    print(f"\n===== LoRA Rank {rank} =====")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto"
    )

    config = LoraConfig(
        r=rank,
        lora_alpha=rank * 2,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, config)

    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    total = sum(
        p.numel() for p in model.parameters()
    )

    percentage = 100 * trainable / total

    rank_results.append({
        "rank": rank,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_%": percentage
    })

    print(f"Trainable params: {trainable:,}")
    print(f"Total params: {total:,}")
    print(f"Trainable %: {percentage:.4f}%")

    del model
    gc.collect()
    torch.cuda.empty_cache()

rank_results


===== LoRA Rank 4 =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable params: 270,336
Total params: 494,303,104
Trainable %: 0.0547%

===== LoRA Rank 8 =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable params: 540,672
Total params: 494,573,440
Trainable %: 0.1093%

===== LoRA Rank 16 =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable params: 1,081,344
Total params: 495,114,112
Trainable %: 0.2184%

===== LoRA Rank 32 =====


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable params: 2,162,688
Total params: 496,195,456
Trainable %: 0.4359%


[{'rank': 4,
  'trainable_params': 270336,
  'total_params': 494303104,
  'trainable_%': 0.05469033024724846},
 {'rank': 8,
  'trainable_params': 540672,
  'total_params': 494573440,
  'trainable_%': 0.1093208725482711},
 {'rank': 16,
  'trainable_params': 1081344,
  'total_params': 495114112,
  'trainable_%': 0.21840298504761665},
 {'rank': 32,
  'trainable_params': 2162688,
  'total_params': 496195456,
  'trainable_%': 0.43585405183557346}]